# Calculo de Tendencias por niveles
***
Created: 17/04/2026                                    updated:01/05/2026


En este notebook se calcula la tendencia por pixeles y para profundides de entre 4000 y el fondo oceánico de forma que cada 20 metros de profundidad se tiene una tendencia. Además, se calculan otros datos necesarios para el calculo del calor, como densidad, area y capacidad calorifica. Finalmente, se calcula también el calor. Por ello, la idea ahora es generar un archivo netcdf que contenga un dataset con las siguientes carácteristicas:
- Dimensiones: (latitud, longitud, presion)
- variables:
    - $tendency \in \mathbb{R}^{lat \times lon\times press}$

Por defecto las fechas en las que se recortan es entre 1990 y 2010, además de profundidades inferiores a 4000 metros, pero esto se puede ajustar al principio.

La lógica del script es la siguiente:
1. Se abren los ficheros grid, join y batimetria
2. Se recortan para las fechas y las profundidades dichas
3. Se crea una función que dada ambos ds recortados devuelva las matrices de latitud, longitud, presión, tendencia media
4. Se guardan dichas matrices en un nuevo fichero netcdf


In [1]:
# Packages for data manipulation
import numpy as np
import xarray as xr
import pandas as pd
import datetime as dt

# Packages for visualization
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Package for progress bar
from tqdm import tqdm

# Package for file handling
import os

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

### Características previas

In [25]:
grid_resolution = 0.25
min_press = 4000
max_press = None
start_year = 1990
end_year = 2025
press_step = 20
code_press =str(min_press).split()[0]

### Apertura y recorte de los archivos

In [26]:
grid_res_str = str(grid_resolution).split('.')[-1]
grid = xr.open_dataset(f'./Data/grid/occupation_grid_{grid_res_str}.nc')
ds = xr.open_dataset('./Data/join/total_filt.nc')

In [27]:
grid

<xarray.Dataset> Size: 1GB
Dimensions:    (latitude: 721, longitude: 1441, n_prof: 78)
Coordinates:
  * latitude   (latitude) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * longitude  (longitude) float64 12kB -180.0 -179.8 -179.5 ... 179.8 180.0
    n          (latitude, longitude) float64 8MB ...
    batimetry  (latitude, longitude) float64 8MB ...
    mask       (latitude, longitude) float64 8MB ...
    basin      (latitude, longitude) <U18 75MB ...
    surface    (latitude, longitude) float64 8MB ...
  * n_prof     (n_prof) int64 624B 0 1 2 3 4 5 6 7 8 ... 70 71 72 73 74 75 76 77
    times      (latitude, longitude, n_prof) datetime64[ns] 648MB ...
Data variables:
    profiles   (latitude, longitude, n_prof) float64 648MB ...

In [28]:
ds

<xarray.Dataset> Size: 6GB
Dimensions:               (N_PROF: 21592, P: 6501)
Coordinates:
    time                  (N_PROF) datetime64[ns] 173kB ...
    section_id            (N_PROF) <U40 3MB ...
    file_name             (N_PROF) <U29 3MB ...
    latitude              (N_PROF) float64 173kB ...
    longitude             (N_PROF) float64 173kB ...
    pressure              (P) int64 52kB ...
Dimensions without coordinates: N_PROF, P
Data variables:
    ctd_temperature_filt  (N_PROF, P) float64 1GB ...
    ctd_salinity_filt     (N_PROF, P) float64 1GB ...
    ctd_oxygen_filt       (N_PROF, P) float64 1GB ...
    ctd_density_filt      (N_PROF, P) float64 1GB ...
    ctd_cp_filt           (N_PROF, P) float64 1GB ...

Recortamos ambos datasets por fechas y profundidades

In [29]:
# Cutting the dataset to have data from the wanted years
grid = grid.where((grid.times.dt.year >= start_year) & (grid.times.dt.year <= end_year), drop = True)

In [30]:
# Cutting the dataset to have data from exact pressure
if max_press is not None:
    ds = ds.where((ds.pressure <= max_press) & (ds.pressure >= min_press), drop=True)

else:
    ds = ds.where(ds.pressure >= min_press, drop=True)

### Función que devuelve tendencias para el nuevo dataset

In [19]:
def tendencias_nivel(ds, grid, press_step = 20):
    # Extract the wanted values
    latitude = grid.latitude.values
    longitude = grid.longitude.values
    pressure = ds.pressure.values
    n = grid.n.values
    profiles = grid.profiles.values
    time_array = ds.time.values

    # Create the array of pressures with the wanted step
    levels = np.arange(np.min(pressure), np.max(pressure), press_step)

    # Array which contain tendency for levels
    tendency_levels = np.full((len(latitude), len(longitude), len(levels)), np.nan)

    # Precalculating the valid indeces
    valid_idx = []
    for i in range(len(latitude)):
        for j in range(len(longitude)):
            # Check if there are profiles
            if not np.isnan(n[i, j]): 
                # Obtaining index of profiles with Nans
                profs = profiles[i, j, :] 
                profs = profs[~np.isnan(profs)].astype(int)

                # Precalculate the dates for this pixel
                dates = time_array[profs]
                valid_idx.append((i, j, profs, dates))

    for k in range(1, len(levels)):
        # Cut the matrix by levels
        ds_level = ds.where((ds.pressure <= levels[k]) & (ds.pressure >= levels[k-1]), drop=True)

        # Calculate the mean for all the profiles
        temp_mean_profs = ds_level.ctd_temperature_filt.mean(dim = 'P').values

        # Explore all the valid data
        for i, j, profs, prof_dates in valid_idx:
            # Extract the mean
            temp_mean = temp_mean_profs[profs]

            # Delete Nans
            valid = ~np.isnan(temp_mean)
            temp_mean = temp_mean[valid]
            valid_dates = prof_dates[valid]

            # Check if there are more than 2 values
            if len(valid_dates) <= 2:
                continue 
            
            # Check if the difference between dates is longer than 2.5 years
            sorted_dates = np.sort(valid_dates)
            if (sorted_dates[-1] - sorted_dates[0]) / np.timedelta64(365, 'D') >= 2.5:
                # Fit to a line curve
                dates_pol = valid_dates.astype('datetime64[ns]').astype(np.int64)
                coefficients = np.polyfit(dates_pol, temp_mean, 1)

                tendency = coefficients[0] / 1.e-9 * 24 * 3600 * 365 * 100 # Convert from ºC/ns to ºC/century

                # Save the value
                tendency_levels[i, j, k] = tendency

                
    return (latitude, longitude, levels, tendency_levels)

Aplicamos y guardamos en un netcdf

In [31]:
# Extract matrix of data
lat, lon, press_levels, tendency_levels = tendencias_nivel(ds = ds, grid = grid, press_step = press_step) 

In [32]:
# Creating the dataset
ds_tendency = xr.Dataset(
    # Variables
    data_vars = {'tendency' : (('latitude', 'longitude', 'pressure'), tendency_levels, {'units': 'ºC/century', 'description': 'Temperature tendency by levels between the asociated pressure value and the one before'})},

    # Coordinates
    coords = {
        'latitude' : (('latitude',), lat ),
        'longitude' : (('longitude',), lon),
        'pressure': (('pressure',), press_levels, {'units' : 'dbar'})},

    # Metadates
    attrs = {'description' : f'Dataset of temperature tendencies by pressure levels between from pressures under {min_press} dbar and for years {start_year}-{end_year} with a spatial resolution of {grid_resolution} and a pressure resolution {press_step} dbar'}
)

# Save it in a NetCDF file
ds_tendency.to_netcdf(f'./Data/tendency_levels/tendency_levels_{start_year}_{end_year}_{str(grid_resolution).split('.')[-1]}_{min_press}_{code_press}.nc')

In [33]:
ds_tendency

<xarray.Dataset> Size: 763MB
Dimensions:    (latitude: 563, longitude: 1355, pressure: 125)
Coordinates:
  * latitude   (latitude) float64 5kB -77.75 -77.5 -77.25 ... 62.75 63.0 63.25
  * longitude  (longitude) float64 11kB -180.0 -179.8 -179.5 ... 179.8 180.0
  * pressure   (pressure) int64 1kB 4000 4020 4040 4060 ... 6420 6440 6460 6480
Data variables:
    tendency   (latitude, longitude, pressure) float64 763MB nan nan ... nan nan
Attributes:
    description:  Dataset of temperature tendencies by pressure levels betwee...

In [34]:
# Close all ds
ds.close()
grid.close()
ds_tendency.close()